In [4]:
model_name = "tiiuae/falcon-rw-1b"

In [ ]:
# 32bit
# 1 sign, 8 mantisssa, 23 bits exponent

-128.75899654899
# 0.24589696325645

# Quantization=memory compression----high-->low(8,4)



# 4-bit=2**4=16=0,,,,,,15
# 0,     4,   8,   12, 15
# -1.0 -0.5, 0.0, 0.5, 1.0

# quantized =( x-min/max-min)/(2**bits-1)



# llama 7b===32bit=4bytes

# 7*4=28gb

# 1.2---2

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="float16")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [6]:
model.get_memory_footprint()/(1024*1024*1024)

5.269973993301392

In [8]:
model.get_memory_footprint()/(1024*1024*1024)

2.6349871158599854

In [ ]:
model.get_memory_footprint()/(1024*1024*1024)

5.269973993301392

In [ ]:
2.6/2

1.3

In [ ]:
1.3/2

0.65

In [9]:
from transformers import BitsAndBytesConfig

In [10]:
import torch

In [11]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True,
                   bnb_4bit_quant_type="nf4",
                   bnb_4bit_use_double_quant=True,
                   bnb_4bit_compute_dtype=torch.float16)

In [13]:
# !pip install -U bitsandbytes

In [14]:
model_q = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [15]:
model_q.get_memory_footprint()/(1024*1024*1024)

0.9474871158599854

In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
lora_config = LoraConfig(r=32,
           lora_alpha=1,#2*r
           target_modules=['query_key_value', "fc1"],
           lora_dropout=0.05,
           bias="lora_only",
           task_type="CAUSAL_LM")

In [ ]:
peft_model = get_peft_model(model_q, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
peft_model.print_trainable_parameters()

trainable params: 6,438,912 || all params: 1,420,939,264 || trainable%: 0.4531
